In [67]:
import fitz  # PyMuPDF
from rank_bm25 import BM25Okapi
from nltk.tokenize import word_tokenize
import nltk
import numpy as np
import pandas as pd
import re

nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [68]:
def tokenize(text):
    text = text.lower()
    tokens = re.findall(r"\b[a-zA-Z]+\b", text)
    return tokens

In [69]:

def extract_pages_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []

    for page_number, page in enumerate(doc, start=1):
        text = page.get_text()

        pages.append({
            "source": Path(pdf_path).name,
            "page": page_number,
            "text": text
        })

    return pages

In [70]:
def chunk_text(text, chunk_size=300, overlap=15):
    words = text.split()
    chunks = []

    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])

        if len(chunk.strip()) > 50:
            chunks.append(chunk)

        start = end - overlap

    return chunks


In [72]:

pdf_files = [
    r"C:\Users\admin\Downloads\NLP lec\LEC\test.pdf",
    r"C:\Users\admin\Downloads\NLP lec\LEC\learning-langchain-for-true-epub-9781098167288.pdf",
    r"C:\Users\admin\Downloads\NLP lec\LEC\Natural-Language-Processing-Python.pdf"
]

all_chunks = []

for pdf_path in pdf_files:
    pages = extract_pages_from_pdf(pdf_path)

    for page in pages:
        chunks = chunk_text(page["text"], chunk_size=300, overlap=15)

        for chunk_id, chunk in enumerate(chunks, start=1):
            all_chunks.append({
                "source": page["source"],
                "page": page["page"],
                "chunk_id": chunk_id,
                "text": chunk
            })

print("Total chunks created:", len(all_chunks))
print(all_chunks[0]["text"][:500])

Total chunks created: 2085
Praise for Hands-On Large Language Models This is an exceptional guide to the world of language models and their practical applications in industry. Its highly-visual coverage of generative, representational, and retrieval applications of language models empowers readers to quickly understand, use, and refine LLMs. Highly recommended! —Nils Reimers, Director of Machine Learning at Cohere | creator of sentence-transformers Jay and Maarten have continued their tradition of providing beautifully il


In [73]:
tokenized_chunks = [
    tokenize(chunk["text"])
    for chunk in all_chunks
]

In [74]:
bm25 = BM25Okapi(tokenized_chunks)

In [76]:
queries = [
        "different kind of tokenization",
        "Different types of tokenization methods"]

for query in queries:
    tokenized_query = tokenize(query)
    scores = bm25.get_scores(tokenized_query)

top_n = 5
top_indices = np.argsort(scores)[::-1][:top_n]

results = []

for rank, idx in enumerate(top_indices, start=1):
    chunk = all_chunks[idx]

    results.append({
        "rank": rank,
        "score": round(float(scores[idx]), 4),
        "source": chunk["source"],
        "page": chunk["page"],
        "chunk_id": chunk["chunk_id"],
        "text_preview": chunk["text"][:700]
    })

df_results = pd.DataFrame(results)
df_results

,rank,score,source,page,chunk_id,text_preview
0,1,14.8752,test.pdf,91,1,1 2 . 0 * 5 0 = 6 0 0 Galactica English and CA...
1,2,13.2945,test.pdf,77,1,Figure 2-6. There are multiple methods of toke...
2,3,13.0110,test.pdf,75,1,"Third, the tokenizer needs to be trained on a ..."
3,4,12.6894,test.pdf,590,1,"creating contextualized word embeddings, Creat..."
4,5,12.0822,test.pdf,147,1,3. Figure 3-25 shows these different types of ...
